# ⚡ Arbitrage Calculator: Enhanced Cloud GPU Matcher v3.1
## 🛡️ BUG FIX: DEDUPLICATION & PURE MATCH VALIDATION

### Fixes implemented:
1. **Input Deduplication**: Ensures we don't match the same market ID multiple times.
2. **Logit Softmaxing**: Converts NLI raw scores to real probabilities (no more '5.45' match scores).
3. **Strict Range Matching**: Requires numerical consistency (won't match 'The Bride!' to 'The Bride! > 90' unless both imply the same threshold).
4. **ROI Capping**: Prevents 'Infinite ROI' hallucinations from zero-priced markets.
5. **Full Title Display**: Removed truncation so you can see the entire question text.

### Instructions:
1. **Runtime > Change runtime type** → Select **T4 GPU**
2. **Runtime > Run all**

In [ ]:
# 1. Install Enhanced Packages
!pip install sentence-transformers torch httpx pydantic dateparser spacy -q
!python -m spacy download en_core_web_sm -q

import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import httpx
import time
import re
import dateparser
import spacy
from datetime import datetime
from typing import List, Dict, Any, Set, Tuple, Optional

print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

# Load spaCy for entity extraction
print("Loading spaCy NLP model...")
nlp = spacy.load("en_core_web_sm")
print("Setup complete!")

In [ ]:
# 2. Load ENHANCED ML Models
print("Loading Bi-Encoder (Fast Semantic Filter)...")
bi_model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda' if torch.cuda.is_available() else 'cpu')

print("\nLoading NLI Cross-Encoder (Logical Equivalence Engine)...")
cross_model = CrossEncoder('cross-encoder/nli-deberta-v3-small', device='cuda' if torch.cuda.is_available() else 'cpu')

print("\n✅ Models loaded! Using NLI for precise semantic matching.")

In [ ]:
# 3. Configure Local Connection
LOCAL_BACKEND_URL = "https://copyrightable-pseudocartilaginous-sade.ngrok-free.dev/api"

def fetch_markets_from_local():
    try:
        print(f"Fetching markets from {LOCAL_BACKEND_URL}/raw-markets...")
        response = httpx.get(f"{LOCAL_BACKEND_URL}/raw-markets", timeout=45.0)
        response.raise_for_status()
        data = response.json()
        
        # DEDUPLICATE INPUT
        seen_ids = set()
        unique_data = []
        for m in data:
            mid = m.get('id', m.get('marketUrl'))
            if mid not in seen_ids:
                seen_ids.add(mid)
                unique_data.append(m)
        return unique_data
    except Exception as e:
        print(f"Could not reach local backend: {e}")
        return []

all_markets = fetch_markets_from_local()
print(f"Fetched {len(all_markets)} UNIQUE markets.")

In [ ]:
# 4. ADVANCED EXTRACTION & COMPARISON LOGIC

def extract_dates(text: str) -> Set[Tuple[int, int, int]]:
    dates = set()
    patterns = [
        r'\b(\d{1,2})/(\d{1,2})/(\d{4})\b',
        r'\b(\d{4})-(\d{1,2})-(\d{1,2})\b',
        r'\b(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{1,2})(?:st|nd|rd|th)?(?:,?\s+(\d{4}))?\b',
    ]
    for pattern in patterns:
        for match in re.finditer(pattern, text, re.IGNORECASE):
            try:
                parsed = dateparser.parse(match.group(0), settings={'PREFER_DATES_FROM': 'future'})
                if parsed: dates.add((parsed.year, parsed.month, parsed.day))
            except: pass
    return dates

def extract_numbers(text: str) -> Set[float]:
    numbers = set()
    # Find all numbers including decimals
    for match in re.finditer(r'\d+(?:\.\d+)?', text):
        num = float(match.group())
        if not (2020 <= num <= 2030): # Exclude years
            numbers.add(num)
    return numbers

def are_questions_compatible(title_a: str, title_b: str) -> Tuple[bool, str]:
    # 1. Date Check
    dates_a, dates_b = extract_dates(title_a), extract_dates(title_b)
    if dates_a and dates_b and dates_a.isdisjoint(dates_b):
        return False, f"Date mismatch"
    
    # 2. Threshold/Number Check (The "The Bride! > 90" Fix)
    nums_a, nums_b = extract_numbers(title_a), extract_numbers(title_b)
    # If one has numbers (thresholds) and the other doesn't, they are NOT "Pure Matches"
    if (nums_a and not nums_b) or (nums_b and not nums_a):
        return False, "Precision mismatch (one question has specific thresholds, other doesn't)"
    
    if nums_a and nums_b:
        if not any(any(abs(na - nb) < 0.1 for nb in nums_b) for na in nums_a):
            return False, "Numerical range mismatch"
    
    # 3. Entity Check
    doc_a, doc_b = nlp(title_a.lower()), nlp(title_b.lower())
    ents_a = {ent.text for ent in doc_a.ents if ent.label_ in ['PERSON', 'ORG', 'GPE']}
    ents_b = {ent.text for ent in doc_b.ents if ent.label_ in ['PERSON', 'ORG', 'GPE']}
    if ents_a and ents_b and ents_a.isdisjoint(ents_b):
        return False, "Core entity mismatch"
        
    return True, "Compatible"

print("✅ Logic updated for strict numerical compliance.")

In [ ]:
# 5. Enhanced Matching + Softmaxed NLI Probs

def compute_pair_arb(ma, mb):
    # Use bid/ask if available, otherwise yesPrice
    p1_a = ma.get('bestBid', ma['yesPrice'])
    p1_b = 1 - mb.get('bestAsk', mb['yesPrice'])
    
    cost1 = p1_a + p1_b
    roi1 = ((1 - cost1) / max(0.01, cost1) * 100) if cost1 < 1 and cost1 > 0.05 else -100
    
    p2_a = 1 - ma.get('bestAsk', ma['yesPrice'])
    p2_b = mb.get('bestBid', mb['yesPrice'])
    
    cost2 = p2_a + p2_b
    roi2 = ((1 - cost2) / max(0.01, cost2) * 100) if cost2 < 1 and cost2 > 0.05 else -100
    
    # CAP ROI at 1000% to avoid div-by-zero glitches
    if roi1 > roi2:
        return {"roi": min(1000, roi1), "cost": cost1, "scenario": 1}
    return {"roi": min(1000, roi2), "cost": cost2, "scenario": 2}

def match_on_gpu(markets, min_bi_sim=0.75, min_roi=0.1, top_k=200):
    by_platform = {}
    for m in markets:
        by_platform.setdefault(m["platform"], []).append(m)
            
    platforms = list(by_platform.keys())
    plat_embeddings = {}
    for plat in platforms:
        titles = [m["title"] for m in by_platform[plat]]
        plat_embeddings[plat] = bi_model.encode(titles, convert_to_tensor=True, batch_size=256)
        
    candidates = []
    for i in range(len(platforms)):
        pa = platforms[i]
        for j in range(i + 1, len(platforms)):
            pb = platforms[j]
            cosine_scores = util.cos_sim(plat_embeddings[pa], plat_embeddings[pb])
            high_score_indices = (cosine_scores >= min_bi_sim).nonzero(as_tuple=False)
            
            for idx in high_score_indices:
                ma, mb = by_platform[pa][idx[0]], by_platform[pb][idx[1]]
                
                compatible, reason = are_questions_compatible(ma['title'], mb['title'])
                if not compatible: continue

                arb = compute_pair_arb(ma, mb)
                if arb["roi"] >= min_roi:
                    candidates.append((ma, mb, arb["roi"], reason))
    
    if not candidates: return []
    
    # RERANK TOP 2000 WITH NLI SOFTMAX
    candidates.sort(key=lambda x: x[2], reverse=True)
    top_candidates = candidates[:2000]
    pairs_to_score = [[c[0]['title'], c[1]['title']] for c in top_candidates]
    
    nli_logits = cross_model.predict(pairs_to_score, show_progress_bar=True)
    final_pairs = []
    
    for i, logits in enumerate(nli_logits):
        ma, mb, roi, compat_reason = top_candidates[i]
        
        # SOFTMAX the logits (Contradiction=0, Neutral=1, Entailment=2)
        probs = F.softmax(torch.tensor(logits), dim=0)
        entail_prob = float(probs[2])
        contra_prob = float(probs[0])
        
        # Score = Entailment Prob - Contradiction Prob (Net Confidence)
        net_score = (entail_prob - contra_prob) * 100
        
        if net_score > 40: # Must be significantly more likely to entail than contradict
            final_pairs.append({
                "marketA": ma, "marketB": mb, "roi": roi,
                "matchScore": round(net_score, 1),
                "matchReason": f"NLI Prob: {entail_prob:.2%} | {compat_reason}",
                "isVerified": entail_prob > 0.85
            })
    
    # FINAL SORT: Verified First, then ROI
    final_pairs.sort(key=lambda x: (x["isVerified"], x["matchScore"], x["roi"]), reverse=True)
    return final_pairs[:top_k]

found_pairs = match_on_gpu(all_markets, min_bi_sim=0.65, min_roi=0.1)
print(f"Found {len(found_pairs)} logically verified matches.")

In [ ]:
# 6. Sync to Local System
def post_results_to_system(pairs):
    if not pairs: return
    print(f"Syncing {len(pairs)} records...")
    try:
        # CLEAR OLD DATA ON FIRST BATCH
        httpx.post(f"{LOCAL_BACKEND_URL}/cloud-results?clear=true", json=pairs[:200], timeout=60.0)
        print("🎉 Successfully updated dashboard with verified matches.")
    except Exception as e:
        print(f"❌ Sync failed: {e}")
        
post_results_to_system(found_pairs)

In [ ]:
# 7. Display Sample Results
print("\n" + "="*90)
print(f"TOP {min(10, len(found_pairs))} PURE OPPORTUNITIES")
print("="*90)
for i, p in enumerate(found_pairs[:10], 1):
    print(f"\n#{i} | ROI: {p['roi']:.2f}% | Quality: {p['matchScore']}% | {'✅ VERIFIED' if p['isVerified'] else '⚠️  Review'}")
    print(f"  [A] {p['marketA']['platform']:11} | {p['marketA']['title']}")
    print(f"  [B] {p['marketB']['platform']:11} | {p['marketB']['title']}")
    print(f"  Reason: {p['matchReason']}")